# Life Insurance Death Claims Analysis

This notebook evaluates insurer-level death-claim settlement quality, rejection and pending risk, market scale, and industry trends using the uploaded CSV. It is designed for business presentation use and adapts to the actual file structure rather than assumed field names.

**Required packages before running:** `pandas`, `plotly`, `nbformat`.


In [15]:
from pathlib import Path
import importlib
import sys
import subprocess

required_packages = ["pandas", "plotly", "nbformat"]
missing = []
for package in required_packages:
    try:
        importlib.import_module(package)
    except ModuleNotFoundError:
        missing.append(package)

if missing:
    print("Installing missing packages:", ", ".join(missing))
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
        importlib.invalidate_caches()
    except Exception as exc:
        raise RuntimeError(
            "Automatic package installation failed. Run this in a notebook cell: %pip install pandas plotly nbformat"
        ) from exc

np = importlib.import_module("numpy")
pd = importlib.import_module("pandas")
px = importlib.import_module("plotly.express")
go = importlib.import_module("plotly.graph_objects")
make_subplots = importlib.import_module("plotly.subplots").make_subplots
pio = importlib.import_module("plotly.io")
display = importlib.import_module("IPython.display").display

pd.set_option("display.max_columns", None)
pd.options.display.float_format = "{:,.2f}".format

pio.templates.default = "plotly_white"
brand = {
    "ink": "#12324A",
    "blue": "#2C7FB8",
    "teal": "#1FA187",
    "amber": "#F4A259",
    "red": "#D95F5F",
    "slate": "#6B7C93",
    "bg": "#F7FAFC",
}


def polish(fig, height=500):
    fig.update_layout(
        height=height,
        paper_bgcolor="white",
        plot_bgcolor=brand["bg"],
        font=dict(family="Aptos, Segoe UI, Arial", color=brand["ink"]),
        margin=dict(l=40, r=30, t=70, b=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    )
    fig.update_xaxes(gridcolor="rgba(18,50,74,0.08)")
    fig.update_yaxes(gridcolor="rgba(18,50,74,0.08)")
    return fig


def pct(series):
    return (series * 100).round(2)


## 1. Data Preparation And Controls

We start by loading the CSV with encoding fallback, removing non-data rows, standardizing insurer aliases, separating aggregate rows from insurer rows, and checking whether row-level totals reconcile to component fields.


In [16]:
candidate_paths = [
    Path("li_death_claims.csv"),
    Path("../li_death_claims.csv"),
    Path.cwd() / "li_death_claims.csv",
    Path.cwd().parent / "li_death_claims.csv",
    Path.cwd() / "14_li_death_claims.csv",
    Path.cwd().parent / "14_li_death_claims.csv",
    Path.cwd() / "14_Data Analysis with AI tools" / "14_li_death_claims.csv",
    Path.cwd().parent / "14_Data Analysis with AI tools" / "14_li_death_claims.csv",
]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not locate li_death_claims.csv in the working directory or parent directory.")

encodings = ["utf-8-sig", "cp1252", "latin1"]
last_error = None
for encoding in encodings:
    try:
        raw_df = pd.read_csv(data_path, encoding=encoding, engine="python", on_bad_lines="skip")
        chosen_encoding = encoding
        break
    except Exception as exc:
        last_error = exc
else:
    raise RuntimeError(f"Unable to read the CSV safely. Last error: {last_error}")

alias_map = {
    "ABSL": "Aditya Birla Life",
    "Ageas": "Ageas Federal",
    "Baj Alz": "Bajaj Allianz",
    "Can HSBC": "Canara HSBC OBC",
    "Edelws": "Edelweiss Tokio",
    "Exide": "Exide Life",
    "Fut Genli": "Future Generali",
    "HDFC": "HDFC Life",
    "ICICI": "ICICI Prudential",
    "Indiafirst": "India First",
    "Industry": "Industry Total",
    "Kotak": "Kotak Mahindra",
    "Max": "Max Life",
    "PNB Metlife": "PNB Met Life",
    "PVT.": "Private Total",
    "Pramerica": "Pramerica Life",
    "Reliance": "Reliance Nippon",
    "SUD": "Star Union",
    "Sahara": "Sahara Life",
}
aggregate_names = {"Industry Total", "Private Total"}

numeric_cols = [
    "claims_pending_start_no", "claims_pending_start_amt", "claims_intimated_no", "claims_intimated_amt",
    "total_claims_no", "total_claims_amt", "claims_paid_no", "claims_paid_amt",
    "claims_repudiated_no", "claims_repudiated_amt", "claims_rejected_no", "claims_rejected_amt",
    "claims_unclaimed_no", "claims_unclaimed_amt", "claims_pending_end_no", "claims_pending_end_amt",
    "claims_paid_ratio_no", "claims_paid_ratio_amt", "claims_repudiated_rejected_ratio_no",
    "claims_repudiated_rejected_ratio_amt", "claims_pending_ratio_no", "claims_pending_ratio_amt",
]

df = raw_df.copy()
df = df.dropna(how="all")
df["life_insurer"] = df["life_insurer"].astype(str).str.strip()
df["year"] = df["year"].astype(str).str.strip()
df = df[df["year"].ne("nan") & df["year"].ne("")]
df = df[~df["life_insurer"].str.contains("http|</pre>|\\x00", case=False, na=False)]
df["life_insurer_std"] = df["life_insurer"].replace(alias_map)
df["is_aggregate"] = df["life_insurer_std"].isin(aggregate_names)
df["category"] = df["category"].fillna("Unknown").astype(str).str.strip()

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["year_start"] = pd.to_numeric(df["year"].str.extract(r"(\d{4})", expand=False), errors="coerce")
df = df.sort_values(["year_start", "life_insurer_std"]).reset_index(drop=True)

df["count_component_total"] = (
    df["claims_paid_no"]
    + df["claims_repudiated_no"]
    + df["claims_rejected_no"]
    + df["claims_unclaimed_no"]
    + df["claims_pending_end_no"]
)
df["amt_component_total"] = (
    df["claims_paid_amt"]
    + df["claims_repudiated_amt"]
    + df["claims_rejected_amt"]
    + df["claims_unclaimed_amt"]
    + df["claims_pending_end_amt"]
)
df["count_diff"] = df["count_component_total"] - df["total_claims_no"]
df["amt_diff"] = df["amt_component_total"] - df["total_claims_amt"]

insurer_df = df.loc[~df["is_aggregate"]].copy()
aggregate_df = df.loc[df["is_aggregate"]].copy()

quality_checks = pd.DataFrame(
    {
        "metric": [
            "Encoding used",
            "Valid cleaned rows",
            "Insurer-level rows",
            "Aggregate rows",
            "Unique insurers",
            "Years covered",
            "Corrupt rows removed",
            "Rows with amount reconciliation gap > 0.5",
        ],
        "value": [
            chosen_encoding,
            len(df),
            len(insurer_df),
            len(aggregate_df),
            insurer_df["life_insurer_std"].nunique(),
            insurer_df["year"].nunique(),
            len(raw_df) - len(df),
            int((df["amt_diff"].abs() > 0.5).sum()),
        ],
    }
)

validation_issues = (
    df.loc[df["amt_diff"].abs() > 0.5]
    .sort_values(["year_start", "life_insurer_std"])
    [["life_insurer_std", "year", "total_claims_amt", "amt_component_total", "amt_diff"]]
)

display(quality_checks)
display(validation_issues.head(12))


,metric,value
0,Encoding used,cp1252
1,Valid cleaned rows,149
2,Insurer-level rows,137
3,Aggregate rows,12
4,Unique insurers,24
5,Years covered,5
6,Corrupt rows removed,2
7,Rows with amount reconciliation gap > 0.5,19


,life_insurer_std,year,total_claims_amt,amt_component_total,amt_diff
0,Aditya Birla Life,2017-18,274.17,283.96,9.79
2,Ageas Federal,2017-18,50.51,52.03,1.53
5,Bharti Axa,2017-18,44.11,45.13,1.02
6,Canara HSBC OBC,2017-18,53.38,54.13,0.75
9,Future Generali,2017-18,40.32,41.17,0.85
10,HDFC Life,2017-18,544.70,554.88,10.19
11,ICICI Prudential,2017-18,777.00,792.37,15.37
12,India First,2017-18,57.62,59.17,1.55
13,Industry Total,2017-18,"15,713.39","15,905.21",191.82
14,Kotak Mahindra,2017-18,134.45,138.53,4.08


### Findings

- The file resolves to **149 clean data rows**, comprising **137 insurer-year rows** and **12 aggregate rows**.
- The dataset covers **24 insurers** across **five years** from **2017-18 to 2021-22**.
- Two trailing non-data rows are removed during ingestion, and several short insurer aliases are consolidated into canonical names.
- Amount totals do not fully reconcile in a subset of rows, especially in **2017-18** and **SBI Life 2021-22**; the notebook flags these gaps without rewriting source values.


## 2. Scale And Market Position

This section compares insurers by claim volume and claim amount, then quantifies how concentrated the market is. Aggregate rows are excluded so insurer rankings are not distorted.


In [17]:
insurer_summary = (
    insurer_df.groupby("life_insurer_std", as_index=False)
    .agg(
        years_covered=("year", "nunique"),
        total_claims_no=("total_claims_no", "sum"),
        total_claims_amt=("total_claims_amt", "sum"),
        claims_paid_no=("claims_paid_no", "sum"),
        claims_paid_amt=("claims_paid_amt", "sum"),
        claims_repudiated_no=("claims_repudiated_no", "sum"),
        claims_repudiated_amt=("claims_repudiated_amt", "sum"),
        claims_rejected_no=("claims_rejected_no", "sum"),
        claims_rejected_amt=("claims_rejected_amt", "sum"),
        claims_pending_end_no=("claims_pending_end_no", "sum"),
        claims_pending_end_amt=("claims_pending_end_amt", "sum"),
    )
)

industry_claims_no = insurer_summary["total_claims_no"].sum()
industry_claims_amt = insurer_summary["total_claims_amt"].sum()

insurer_summary["share_of_claims_no"] = insurer_summary["total_claims_no"] / industry_claims_no
insurer_summary["share_of_claims_amt"] = insurer_summary["total_claims_amt"] / industry_claims_amt
insurer_summary["avg_claim_size"] = insurer_summary["total_claims_amt"] / insurer_summary["total_claims_no"]

top_count = insurer_summary.sort_values("total_claims_no", ascending=False).head(10).copy()
top_amount = insurer_summary.sort_values("total_claims_amt", ascending=False).head(10).copy()
concentration = insurer_summary.sort_values("share_of_claims_no", ascending=False).head(10).copy()

display(top_count[["life_insurer_std", "total_claims_no", "share_of_claims_no"]].assign(share_of_claims_no=pct(top_count["share_of_claims_no"])))
display(top_amount[["life_insurer_std", "total_claims_amt", "share_of_claims_amt"]].assign(share_of_claims_amt=pct(top_amount["share_of_claims_amt"])))

fig_count = px.bar(
    top_count.sort_values("total_claims_no"),
    x="total_claims_no",
    y="life_insurer_std",
    orientation="h",
    text="total_claims_no",
    color_discrete_sequence=[brand["blue"]],
    title="Top Insurers By Claim Count",
)
fig_count.update_traces(texttemplate="%{text:,.0f}", hovertemplate="%{y}<br>Claims: %{x:,.0f}<extra></extra>")
polish(fig_count, height=520).show()

fig_amt = px.bar(
    top_amount.sort_values("total_claims_amt"),
    x="total_claims_amt",
    y="life_insurer_std",
    orientation="h",
    text="total_claims_amt",
    color_discrete_sequence=[brand["teal"]],
    title="Top Insurers By Claim Amount",
)
fig_amt.update_traces(texttemplate="%{text:,.0f}", hovertemplate="%{y}<br>Claim amount: %{x:,.2f}<extra></extra>")
polish(fig_amt, height=520).show()

fig_share = px.bar(
    concentration.sort_values("share_of_claims_no"),
    x="share_of_claims_no",
    y="life_insurer_std",
    orientation="h",
    text=concentration.sort_values("share_of_claims_no")["share_of_claims_no"].map(lambda x: f"{x:.1%}"),
    color_discrete_sequence=[brand["amber"]],
    title="Market Concentration By Claim Count",
)
fig_share.update_traces(hovertemplate="%{y}<br>Claim share: %{x:.2%}<extra></extra>")
fig_share.update_xaxes(tickformat=".0%")
polish(fig_share, height=520).show()


,life_insurer_std,total_claims_no,share_of_claims_no
14,LIC,"4,563,028.00",84.43
19,SBI Life,"151,282.00",2.80
15,Max Life,"112,021.00",2.07
10,HDFC Life,"99,212.00",1.84
4,Bajaj Allianz,"88,925.00",1.65
11,ICICI Prudential,"85,684.00",1.59
18,Reliance Nippon,"57,937.00",1.07
0,Aditya Birla Life,"38,869.00",0.72
16,PNB Met Life,"31,839.00",0.59
8,Exide Life,"27,543.00",0.51


,life_insurer_std,total_claims_amt,share_of_claims_amt
14,LIC,"87,268.16",64.00
11,ICICI Prudential,"9,279.49",6.81
10,HDFC Life,"7,384.19",5.42
19,SBI Life,"6,840.36",5.02
15,Max Life,"5,379.37",3.94
4,Bajaj Allianz,"2,812.91",2.06
0,Aditya Birla Life,"2,777.74",2.04
23,Tata AIA,"2,673.19",1.96
16,PNB Met Life,"2,138.82",1.57
13,Kotak Mahindra,"1,717.61",1.26


### Findings

- **LIC** is the dominant carrier by a wide margin, accounting for roughly **84.4% of claim count** and **64.0% of claim amount** across the cleaned insurer-level data.
- By claim count, the next largest insurers are **SBI Life**, **Max Life**, **HDFC Life**, and **Bajaj Allianz**, but all remain far behind LIC.
- By claim amount, **ICICI Prudential** and **HDFC Life** move up the ranking, indicating larger average ticket sizes than several volume-led peers.
- The industry is structurally concentrated, so headline market averages are heavily influenced by LIC rather than by the median private insurer.


## 3. Insurer Settlement Quality

Settlement quality is scored with a weighted framework that emphasizes paid ratios while penalizing repudiation, rejection, and pending closure drag. A credibility adjustment shrinks scores for small books so low-volume insurers do not appear artificially strong.


In [18]:
insurer_summary["paid_rate_no"] = insurer_summary["claims_paid_no"] / insurer_summary["total_claims_no"]
insurer_summary["paid_rate_amt"] = insurer_summary["claims_paid_amt"] / insurer_summary["total_claims_amt"]
insurer_summary["rep_rej_rate_no"] = (
    insurer_summary["claims_repudiated_no"] + insurer_summary["claims_rejected_no"]
) / insurer_summary["total_claims_no"]
insurer_summary["rep_rej_rate_amt"] = (
    insurer_summary["claims_repudiated_amt"] + insurer_summary["claims_rejected_amt"]
) / insurer_summary["total_claims_amt"]
insurer_summary["pending_rate_no"] = insurer_summary["claims_pending_end_no"] / insurer_summary["total_claims_no"]
insurer_summary["pending_rate_amt"] = insurer_summary["claims_pending_end_amt"] / insurer_summary["total_claims_amt"]

insurer_summary["credibility"] = np.clip(np.log10(insurer_summary["total_claims_no"] + 1) / 4, 0, 1)
insurer_summary["quality_score_raw"] = (
    0.45 * insurer_summary["paid_rate_no"]
    + 0.20 * insurer_summary["paid_rate_amt"]
    + 0.20 * (1 - insurer_summary["rep_rej_rate_no"])
    + 0.10 * (1 - insurer_summary["pending_rate_no"])
    + 0.05 * (1 - insurer_summary["pending_rate_amt"])
)
insurer_summary["settlement_quality_score"] = 100 * (
    insurer_summary["credibility"] * insurer_summary["quality_score_raw"]
    + (1 - insurer_summary["credibility"]) * 0.75
)

eligibility_threshold = 5000
eligible = insurer_summary.loc[
    (insurer_summary["total_claims_no"] >= eligibility_threshold)
    & (insurer_summary["years_covered"] >= 3)
].copy()

strong_insurers = eligible.sort_values("settlement_quality_score", ascending=False).head(5).copy()
concerning_insurers = eligible.sort_values("settlement_quality_score", ascending=True).head(5).copy()
best_insurer = strong_insurers.iloc[0]

display(
    strong_insurers[
        [
            "life_insurer_std",
            "settlement_quality_score",
            "total_claims_no",
            "paid_rate_no",
            "rep_rej_rate_no",
            "pending_rate_no",
        ]
    ].assign(
        paid_rate_no=pct(strong_insurers["paid_rate_no"]),
        rep_rej_rate_no=pct(strong_insurers["rep_rej_rate_no"]),
        pending_rate_no=pct(strong_insurers["pending_rate_no"]),
    )
)
display(
    concerning_insurers[
        [
            "life_insurer_std",
            "settlement_quality_score",
            "total_claims_no",
            "paid_rate_no",
            "rep_rej_rate_no",
            "pending_rate_no",
        ]
    ].assign(
        paid_rate_no=pct(concerning_insurers["paid_rate_no"]),
        rep_rej_rate_no=pct(concerning_insurers["rep_rej_rate_no"]),
        pending_rate_no=pct(concerning_insurers["pending_rate_no"]),
    )
)

fig_quality = px.scatter(
    eligible,
    x="rep_rej_rate_no",
    y="settlement_quality_score",
    size="total_claims_no",
    color="pending_rate_no",
    hover_name="life_insurer_std",
    color_continuous_scale=[[0, brand["teal"]], [0.5, brand["amber"]], [1, brand["red"]]],
    title="Settlement Quality Versus Repudiation And Rejection",
    labels={
        "rep_rej_rate_no": "Repudiation + rejection rate",
        "settlement_quality_score": "Settlement quality score",
        "pending_rate_no": "Pending rate",
    },
)
fig_quality.update_xaxes(tickformat=".1%")
polish(fig_quality, height=560).show()


,life_insurer_std,settlement_quality_score,total_claims_no,paid_rate_no,rep_rej_rate_no,pending_rate_no
15,Max Life,98.58,"112,021.00",99.15,0.85,0.01
14,LIC,97.88,"4,563,028.00",98.10,1.11,0.25
5,Bharti Axa,97.73,"8,369.00",98.34,1.45,0.22
23,Tata AIA,97.67,"21,249.00",98.49,1.49,0.02
8,Exide Life,97.35,"27,543.00",98.24,0.93,0.81


,life_insurer_std,settlement_quality_score,total_claims_no,paid_rate_no,rep_rej_rate_no,pending_rate_no
21,Shriram,89.64,"18,601.00",90.75,8.60,0.53
9,Future Generali,93.67,"7,707.00",94.95,4.45,0.45
2,Ageas Federal,94.62,"10,187.00",95.52,3.08,1.41
12,India First,94.83,"17,192.00",95.57,4.08,0.38
19,SBI Life,95.34,"151,282.00",95.48,3.59,0.67


### Findings

- On the credibility-weighted score, **Max Life** is the strongest insurer in this dataset, combining scale with a very high paid ratio and near-zero pending drag.
- The leading tier is **Max Life**, **LIC**, **Bharti Axa**, **Tata AIA**, and **Exide Life** after applying the minimum-volume screen.
- The most concerning eligible insurers are **Shriram**, **Future Generali**, **Ageas Federal**, **India First**, and **SBI Life**, driven by weaker closure quality rather than scale alone.
- High settlement quality is consistently associated with low repudiation or rejection rates; pending levels matter, but they are a secondary differentiator in this file.


## 4. Repudiation, Rejection, And Pending Risk

This view isolates the operational friction points in claim settlement. The emphasis is on weighted insurer performance, not single-year outliers, so the comparison reflects recurring execution quality.


In [19]:
risk_view = eligible[
    [
        "life_insurer_std",
        "total_claims_no",
        "rep_rej_rate_no",
        "rep_rej_rate_amt",
        "pending_rate_no",
        "pending_rate_amt",
        "paid_rate_no",
    ]
].copy()

highest_rep_rej = risk_view.sort_values("rep_rej_rate_no", ascending=False).head(10)
highest_pending = risk_view.sort_values("pending_rate_no", ascending=False).head(10)

display(
    highest_rep_rej.assign(
        rep_rej_rate_no=pct(highest_rep_rej["rep_rej_rate_no"]),
        rep_rej_rate_amt=pct(highest_rep_rej["rep_rej_rate_amt"]),
        pending_rate_no=pct(highest_rep_rej["pending_rate_no"]),
        pending_rate_amt=pct(highest_rep_rej["pending_rate_amt"]),
        paid_rate_no=pct(highest_rep_rej["paid_rate_no"]),
    )
)

fig_rep = px.bar(
    highest_rep_rej.sort_values("rep_rej_rate_no"),
    x="rep_rej_rate_no",
    y="life_insurer_std",
    orientation="h",
    text=highest_rep_rej.sort_values("rep_rej_rate_no")["rep_rej_rate_no"].map(lambda x: f"{x:.1%}"),
    color_discrete_sequence=[brand["red"]],
    title="Highest Repudiation And Rejection Rates",
)
fig_rep.update_xaxes(tickformat=".0%")
polish(fig_rep, height=520).show()

fig_pending = px.bar(
    highest_pending.sort_values("pending_rate_no"),
    x="pending_rate_no",
    y="life_insurer_std",
    orientation="h",
    text=highest_pending.sort_values("pending_rate_no")["pending_rate_no"].map(lambda x: f"{x:.1%}"),
    color_discrete_sequence=[brand["amber"]],
    title="Highest Pending Claim Rates",
)
fig_pending.update_xaxes(tickformat=".0%")
polish(fig_pending, height=520).show()


,life_insurer_std,total_claims_no,rep_rej_rate_no,rep_rej_rate_amt,pending_rate_no,pending_rate_amt,paid_rate_no
21,Shriram,"18,601.00",8.60,20.50,0.53,0.90,90.75
9,Future Generali,"7,707.00",4.45,9.74,0.45,2.26,94.95
12,India First,"17,192.00",4.08,8.26,0.38,2.45,95.57
19,SBI Life,"151,282.00",3.59,6.17,0.67,1.75,95.48
22,Star Union,"9,968.00",3.46,7.46,0.31,0.79,96.16
16,PNB Met Life,"31,839.00",3.31,8.34,0.01,0.12,96.65
2,Ageas Federal,"10,187.00",3.08,7.81,1.41,4.40,95.52
3,Aviva,"5,593.00",2.72,3.39,0.16,0.21,97.01
4,Bajaj Allianz,"88,925.00",2.64,6.93,0.03,0.64,97.01
6,Canara HSBC OBC,"9,730.00",2.26,4.55,0.59,2.02,97.14


### Findings

- **Shriram** stands out as the clearest operational concern, with materially weaker paid performance and the highest repudiation or rejection pressure among meaningful-volume insurers.
- **Future Generali**, **India First**, and **Ageas Federal** also sit on the weaker side of the market because repudiation or rejection remains elevated relative to peers.
- Pending claim rates are generally low across the industry, so repudiation and rejection are the more important source of insurer differentiation.
- Where pending ratios do rise, they tend to reinforce an already weaker settlement profile rather than create the problem on their own.


## 5. Industry Trend Across Years

The final analytical section tracks how overall death-claim volume, claim amount, and settlement quality moved from 2017-18 through 2021-22. The objective is to identify whether the industry is improving at scale or simply growing larger.


In [20]:
yearly = (
    insurer_df.groupby(["year_start", "year"], as_index=False)
    .agg(
        total_claims_no=("total_claims_no", "sum"),
        total_claims_amt=("total_claims_amt", "sum"),
        claims_paid_no=("claims_paid_no", "sum"),
        claims_paid_amt=("claims_paid_amt", "sum"),
        claims_repudiated_no=("claims_repudiated_no", "sum"),
        claims_rejected_no=("claims_rejected_no", "sum"),
        claims_pending_end_no=("claims_pending_end_no", "sum"),
    )
    .sort_values("year_start")
)

yearly["paid_rate_no"] = yearly["claims_paid_no"] / yearly["total_claims_no"]
yearly["rep_rej_rate_no"] = (yearly["claims_repudiated_no"] + yearly["claims_rejected_no"]) / yearly["total_claims_no"]
yearly["pending_rate_no"] = yearly["claims_pending_end_no"] / yearly["total_claims_no"]

lic_yearly = (
    insurer_df.loc[insurer_df["life_insurer_std"].eq("LIC")]
    .groupby(["year_start", "year"], as_index=False)
    .agg(total_claims_no=("total_claims_no", "sum"), total_claims_amt=("total_claims_amt", "sum"))
    .sort_values("year_start")
)
yearly = yearly.merge(
    lic_yearly.rename(columns={"total_claims_no": "lic_claims_no", "total_claims_amt": "lic_claims_amt"}),
    on=["year_start", "year"],
    how="left",
)
yearly["lic_share_no"] = yearly["lic_claims_no"] / yearly["total_claims_no"]
yearly["lic_share_amt"] = yearly["lic_claims_amt"] / yearly["total_claims_amt"]

display(
    yearly[
        [
            "year",
            "total_claims_no",
            "total_claims_amt",
            "paid_rate_no",
            "rep_rej_rate_no",
            "pending_rate_no",
            "lic_share_no",
            "lic_share_amt",
        ]
    ].assign(
        paid_rate_no=pct(yearly["paid_rate_no"]),
        rep_rej_rate_no=pct(yearly["rep_rej_rate_no"]),
        pending_rate_no=pct(yearly["pending_rate_no"]),
        lic_share_no=pct(yearly["lic_share_no"]),
        lic_share_amt=pct(yearly["lic_share_amt"]),
    )
)

fig_trend = make_subplots(specs=[[{"secondary_y": True}]])
fig_trend.add_trace(
    go.Scatter(
        x=yearly["year"],
        y=yearly["total_claims_no"],
        mode="lines+markers",
        name="Claim count",
        line=dict(color=brand["blue"], width=3),
    ),
    secondary_y=False,
)
fig_trend.add_trace(
    go.Scatter(
        x=yearly["year"],
        y=yearly["total_claims_amt"],
        mode="lines+markers",
        name="Claim amount",
        line=dict(color=brand["teal"], width=3),
    ),
    secondary_y=True,
)
fig_trend.update_yaxes(title_text="Claim count", secondary_y=False)
fig_trend.update_yaxes(title_text="Claim amount", secondary_y=True)
fig_trend.update_layout(title="Industry Growth In Claim Count And Claim Amount")
polish(fig_trend, height=500).show()

fig_rates = px.line(
    yearly,
    x="year",
    y=["paid_rate_no", "rep_rej_rate_no", "pending_rate_no"],
    markers=True,
    color_discrete_map={
        "paid_rate_no": brand["teal"],
        "rep_rej_rate_no": brand["red"],
        "pending_rate_no": brand["amber"],
    },
    title="Industry Settlement Rates Over Time",
)
fig_rates.update_yaxes(tickformat=".0%")
fig_rates.update_layout(legend_title_text="")
polish(fig_rates, height=500).show()


,year,total_claims_no,total_claims_amt,paid_rate_no,rep_rej_rate_no,pending_rate_no,lic_share_no,lic_share_amt
0,2017-18,"847,986.00","15,713.39",97.68,2.17,0.12,87.16,72.42
1,2018-19,"863,237.00","18,423.52",97.64,1.17,0.12,86.99,73.29
2,2019-20,"874,849.00","19,419.71",96.76,1.28,0.71,86.75,70.52
3,2020-21,"1,209,736.00","34,725.27",98.38,1.16,0.29,78.28,55.02
4,2021-22,"1,608,924.00","48,080.54",98.64,1.03,0.16,84.97,61.53


### Findings

- Industry claim count rises from roughly **0.85 million** in **2017-18** to **1.61 million** in **2021-22**, while claim amount expands from about **15.7k** to **48.1k** in the same period.
- Settlement performance remains strong overall and improves after the temporary deterioration visible in **2019-20**.
- By **2021-22**, the industry paid ratio reaches about **98.6% by count**, while repudiation or rejection and pending ratios both move lower.
- LIC remains the structural anchor each year, meaning industry trend lines reflect both genuine market growth and sustained concentration in one dominant institution.


## 6. Business Conclusion

- **Best insurer on death-claim settlement quality:** Max Life.
- **Top strong insurers:** Max Life, LIC, Bharti Axa, Tata AIA, Exide Life.
- **Top concerning insurers:** Shriram, Future Generali, Ageas Federal, India First, SBI Life.
- **Top by claim count:** LIC, SBI Life, Max Life, HDFC Life, Bajaj Allianz.
- **Top by claim amount:** LIC, ICICI Prudential, HDFC Life, SBI Life, Max Life.
- **Yearly industry trend summary:** the market grows sharply in both volume and value over five years, with paid ratios remaining high and improving after a 2019-20 dip.
- **Major structural takeaway:** this is a highly concentrated industry. LIC alone shapes most of the market-level view, especially on claim count, so insurer benchmarking must be done at carrier level rather than from industry averages alone.

For a business audience, the core message is straightforward: the sector is scaling rapidly and overall settlement quality is healthy, but insurer performance is not uniform. Max Life and a small set of peers stand out for strong execution, while Shriram and several mid-tier carriers show materially weaker closure quality. Any distribution, partnership, or benchmarking decision should therefore balance settlement quality with scale and explicitly separate LIC-driven industry effects from private-insurer operating performance.
